In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


collect monthly state-level oil prices: loop thru available datasets and concat

In [63]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import io

In [64]:
url = "https://www.eia.gov/dnav/pet/pet_pri_dfp1_k_m.htm"
page = requests.get(url)
soup = BeautifulSoup(page.text, "html.parser")

exclude_codes = ["F009960", "F001236", "F003075", "F005071", "F005061", "F005074"]

codes = []
for link in soup.find_all("a"):
    href = link.get("href", "")
    text = link.text.strip()
    if "s=F" in href and "&f=M" in href:
        code = href.split("s=")[-1].split("&")[0]
        num_part = int(code[1:7])
        if num_part % 1000 == 0:
          continue  # skip aggregates
        if code[:7] in exclude_codes:
          continue  # skip unwanted states
        codes.append(code)
print(codes)

['F001242__3', 'F001354__3', 'F002017__3', 'F002018__3', 'F002020__3', 'F002021__3', 'F002026__3', 'F002031__3', 'F002038__3', 'F002039__3', 'F002040__3', 'F002046__3', 'F003001__3', 'F003005__3', 'F003022__3', 'F003028__3', 'F003035__3', 'F003048__3', 'F004008__3', 'F004030__3', 'F004049__3', 'F004056__3', 'F005006__3']


In [65]:
all_data = []
API_URL = "***YOUR API KEY HERE***"

for code in codes:
    params = {
        "api_key": "vUc8dcxSykilX0CpePaERE6F0NRipDBTqiB1Jl51",
        "frequency": "monthly",
        "data[0]": "value",
        "facets[series][]": code,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": 0,
        "length": 5000
    }
    try:
      response = requests.get(API_URL, params=params)
      response.raise_for_status()
      data = response.json()

      # Extract records
      for r in data['response']['data']:
          all_data.append({
              "State_Code": code,
              "Period": r['period'],
              "Price": r['value']
          })
    except Exception as e:
      print(f"Error fetching data for code {code}: {e}")

# Convert to DataFrame and process as before
df = pd.DataFrame(all_data)
df['Period'] = pd.to_datetime(df['Period'])
df['Year'] = df['Period'].dt.year
df['Month'] = df['Period'].dt.month
df = df[(df['Year'] >= 1978) & (df['Year'] <= 2024)]
df.sort_values(by=['State_Code', 'Year', 'Month'], inplace=True)

df.head(10)

,State_Code,Period,Price,Year,Month
572,F001242__3,1978-01-01,14.71,1978,1
571,F001242__3,1978-02-01,14.8,1978,2
570,F001242__3,1978-03-01,14.74,1978,3
569,F001242__3,1978-04-01,14.67,1978,4
568,F001242__3,1978-05-01,14.71,1978,5
567,F001242__3,1978-06-01,14.72,1978,6
566,F001242__3,1978-07-01,14.75,1978,7
565,F001242__3,1978-08-01,14.65,1978,8
564,F001242__3,1978-09-01,14.7,1978,9
563,F001242__3,1978-10-01,14.78,1978,10


In [66]:
# data cleaning: remove rows where price is withdrawn
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df = df.dropna(subset=['Price'])
df = df.reset_index(drop=True)

In [67]:
# add state column
state_mapping = {
    'F001242__3': 'PA',
    'F001354__3': 'WV',
    'F002017__3': 'IL',
    'F002018__3': 'IN',
    'F002020__3': 'KS',
    'F002021__3': 'KY',
    'F002026__3': 'MI',
    'F002031__3': 'NE',
    'F002038__3': 'ND',
    'F002039__3': 'OH',
    'F002040__3': 'OK',
    'F002046__3': 'SD',
    'F003001__3': 'AL',
    'F003005__3': 'AR',
    'F003022__3': 'LA',
    'F003028__3': 'MS',
    'F003035__3': 'NM',
    'F003048__3': 'TX',
    'F004008__3': 'CO',
    'F004030__3': 'MT',
    'F004049__3': 'UT',
    'F004056__3': 'WY',
    'F005006__3': 'CA'
}

df['state'] = df['State_Code'].map(state_mapping)
print(df.head(10))
print(df.shape)

   State_Code     Period  Price  Year  Month state
0  F001242__3 1978-01-01  14.71  1978      1    PA
1  F001242__3 1978-02-01  14.80  1978      2    PA
2  F001242__3 1978-03-01  14.74  1978      3    PA
3  F001242__3 1978-04-01  14.67  1978      4    PA
4  F001242__3 1978-05-01  14.71  1978      5    PA
5  F001242__3 1978-06-01  14.72  1978      6    PA
6  F001242__3 1978-07-01  14.75  1978      7    PA
7  F001242__3 1978-08-01  14.65  1978      8    PA
8  F001242__3 1978-09-01  14.70  1978      9    PA
9  F001242__3 1978-10-01  14.78  1978     10    PA
(12784, 6)


In [68]:
df.to_csv('/content/drive/MyDrive/agg_oil_price_data.csv', index=False)

Integrate electricity consumption data

In [69]:
cons_csv = pd.read_csv('/content/drive/MyDrive/consumption_monthly.csv')
cons_csv.head(20)

,YEAR,MONTH,STATE,TYPE OF PRODUCER,ENERGY SOURCE (UNITS),CONSUMPTION
0,2001,1,AK,Total Electric Power Industry,Coal (Short Tons),"47,615"
1,2001,1,AK,Total Electric Power Industry,Petroleum (Barrels),"124,998"
2,2001,1,AK,Total Electric Power Industry,Natural Gas (Mcf),"3,941,267"
3,2001,1,AK,"Electric Generators, Electric Utilities",Coal (Short Tons),"16,535"
4,2001,1,AK,"Electric Generators, Electric Utilities",Petroleum (Barrels),"114,198"
5,2001,1,AK,"Electric Generators, Electric Utilities",Natural Gas (Mcf),"3,189,447"
6,2001,1,AK,"Combined Heat and Power, Electric Power",Coal (Short Tons),"22,890"
7,2001,1,AK,"Combined Heat and Power, Electric Power",Petroleum (Barrels),550
8,2001,1,AK,"Combined Heat and Power, Commercial Power",Coal (Short Tons),"8,190"
9,2001,1,AK,"Combined Heat and Power, Commercial Power",Petroleum (Barrels),"1,590"


In [70]:
# only include petroleum quantities, sum by year - state combination
cons_csv = cons_csv[(cons_csv['ENERGY SOURCE              (UNITS)'].str.contains('Petroleum')) & (cons_csv['TYPE OF PRODUCER']=='Total Electric Power Industry')]
print(cons_csv.head())
print(cons_csv.shape)

    YEAR MONTH STATE               TYPE OF PRODUCER  \
1   2001     1    AK  Total Electric Power Industry   
13  2001     1    AL  Total Electric Power Industry   
29  2001     1    AR  Total Electric Power Industry   
39  2001     1    AZ  Total Electric Power Industry   
51  2001     1    CA  Total Electric Power Industry   

   ENERGY SOURCE              (UNITS) CONSUMPTION  
1                 Petroleum (Barrels)     124,998  
13                Petroleum (Barrels)     284,241  
29                Petroleum (Barrels)     209,194  
39                Petroleum (Barrels)     268,016  
51                Petroleum (Barrels)     960,824  
(15392, 6)


In [72]:
# join the two tables
cons_csv['YEAR'] = cons_csv['YEAR'].astype('int32')
cons_csv['MONTH'] = cons_csv['MONTH'].astype('int32')
df = df.merge(cons_csv, left_on=['Year', 'state', 'Month'], right_on=['YEAR', 'STATE', 'MONTH'], how='inner')
print(df.head())
print(df.shape)

   State_Code     Period  Price  Year  Month state  YEAR  MONTH STATE  \
0  F001242__3 2001-01-01  28.02  2001      1    PA  2001      1    PA   
1  F001242__3 2001-02-01  28.52  2001      2    PA  2001      2    PA   
2  F001242__3 2001-03-01  26.20  2001      3    PA  2001      3    PA   
3  F001242__3 2001-04-01  26.31  2001      4    PA  2001      4    PA   
4  F001242__3 2001-05-01  27.43  2001      5    PA  2001      5    PA   

                TYPE OF PRODUCER ENERGY SOURCE              (UNITS)  \
0  Total Electric Power Industry                Petroleum (Barrels)   
1  Total Electric Power Industry                Petroleum (Barrels)   
2  Total Electric Power Industry                Petroleum (Barrels)   
3  Total Electric Power Industry                Petroleum (Barrels)   
4  Total Electric Power Industry                Petroleum (Barrels)   

  CONSUMPTION  
0   1,005,652  
1     196,117  
2     478,840  
3     960,097  
4     524,322  
(6499, 12)


In [73]:
df = df.drop(columns=['YEAR', 'STATE', 'MONTH'])
df.to_csv('/content/drive/MyDrive/agg_oil_electricity_data.csv', index=False)

Now integrate confounders: interest rate, industrial index, inflation rate, mean temp, var temp, electricity price

In [74]:
# integrate yield curve slope (10Y - 2Y spread)
yield_curve = pd.read_csv('/content/T10Y2YM.csv')
yield_curve.head()

,observation_date,T10Y2YM
0,1976-06-01,0.80
1,1976-07-01,0.98
2,1976-08-01,1.14
3,1976-09-01,1.17
4,1976-10-01,1.43


In [75]:
yield_curve.rename(columns={'T10Y2YM':'Treasury Yield Spread (10Y-2Y)'}, inplace=True)
yield_curve['Year'] = yield_curve['observation_date'].str.split('-').str[0]
yield_curve['Month'] = yield_curve['observation_date'].str.split('-').str[1]
yield_curve.drop(columns=['observation_date'], inplace=True)
yield_curve['Year'] = yield_curve['Year'].astype('int32')
yield_curve['Month'] = yield_curve['Month'].astype('int32')

df = df.merge(yield_curve, on=['Year', 'Month'], how='inner')
print(df.head())
print(df.shape)

   State_Code     Period  Price  Year  Month state  \
0  F001242__3 2001-01-01  28.02  2001      1    PA   
1  F001242__3 2001-02-01  28.52  2001      2    PA   
2  F001242__3 2001-03-01  26.20  2001      3    PA   
3  F001242__3 2001-04-01  26.31  2001      4    PA   
4  F001242__3 2001-05-01  27.43  2001      5    PA   

                TYPE OF PRODUCER ENERGY SOURCE              (UNITS)  \
0  Total Electric Power Industry                Petroleum (Barrels)   
1  Total Electric Power Industry                Petroleum (Barrels)   
2  Total Electric Power Industry                Petroleum (Barrels)   
3  Total Electric Power Industry                Petroleum (Barrels)   
4  Total Electric Power Industry                Petroleum (Barrels)   

  CONSUMPTION  Treasury Yield Spread (10Y-2Y)  
0   1,005,652                            0.40  
1     196,117                            0.44  
2     478,840                            0.55  
3     960,097                            0.91  
4     52

In [76]:
inflation = pd.read_csv('/content/FPCPITOTLZGUSA.csv')
inflation.head()

,observation_date,FPCPITOTLZGUSA
0,1960-01-01,1.457976
1,1961-01-01,1.070724
2,1962-01-01,1.198773
3,1963-01-01,1.239669
4,1964-01-01,1.278912


In [77]:
inflation.rename(columns={'FPCPITOTLZGUSA':'Inflation Rate (%)'}, inplace=True)
inflation['Year'] = inflation['observation_date'].str.split('-').str[0]
inflation.drop(columns=['observation_date'], inplace=True)
inflation['Year'] = inflation['Year'].astype('int32')

df = df.merge(inflation, on='Year', how='inner')
print(df.head())
print(df.shape)

   State_Code     Period  Price  Year  Month state  \
0  F001242__3 2001-01-01  28.02  2001      1    PA   
1  F001242__3 2001-02-01  28.52  2001      2    PA   
2  F001242__3 2001-03-01  26.20  2001      3    PA   
3  F001242__3 2001-04-01  26.31  2001      4    PA   
4  F001242__3 2001-05-01  27.43  2001      5    PA   

                TYPE OF PRODUCER ENERGY SOURCE              (UNITS)  \
0  Total Electric Power Industry                Petroleum (Barrels)   
1  Total Electric Power Industry                Petroleum (Barrels)   
2  Total Electric Power Industry                Petroleum (Barrels)   
3  Total Electric Power Industry                Petroleum (Barrels)   
4  Total Electric Power Industry                Petroleum (Barrels)   

  CONSUMPTION  Treasury Yield Spread (10Y-2Y)  Inflation Rate (%)  
0   1,005,652                            0.40            2.826171  
1     196,117                            0.44            2.826171  
2     478,840                            0.5

In [78]:
ind_idx = pd.read_csv('/content/INDPRO.csv')
ind_idx.head()

,observation_date,INDPRO
0,1919-01-01,4.8739
1,1919-02-01,4.6585
2,1919-03-01,4.5238
3,1919-04-01,4.6046
4,1919-05-01,4.6315


In [79]:
ind_idx.rename(columns={'INDPRO':'Industrial Production Index'}, inplace=True)
ind_idx['Year'] = ind_idx['observation_date'].str.split('-').str[0]
ind_idx['Month'] = ind_idx['observation_date'].str.split('-').str[1]
ind_idx.drop(columns=['observation_date'], inplace=True)
ind_idx['Year'] = ind_idx['Year'].astype('int32')
ind_idx['Month'] = ind_idx['Month'].astype('int32')

df = df.merge(ind_idx, on=['Year', 'Month'], how='inner')
print(df.head())
print(df.shape)

   State_Code     Period  Price  Year  Month state  \
0  F001242__3 2001-01-01  28.02  2001      1    PA   
1  F001242__3 2001-02-01  28.52  2001      2    PA   
2  F001242__3 2001-03-01  26.20  2001      3    PA   
3  F001242__3 2001-04-01  26.31  2001      4    PA   
4  F001242__3 2001-05-01  27.43  2001      5    PA   

                TYPE OF PRODUCER ENERGY SOURCE              (UNITS)  \
0  Total Electric Power Industry                Petroleum (Barrels)   
1  Total Electric Power Industry                Petroleum (Barrels)   
2  Total Electric Power Industry                Petroleum (Barrels)   
3  Total Electric Power Industry                Petroleum (Barrels)   
4  Total Electric Power Industry                Petroleum (Barrels)   

  CONSUMPTION  Treasury Yield Spread (10Y-2Y)  Inflation Rate (%)  \
0   1,005,652                            0.40            2.826171   
1     196,117                            0.44            2.826171   
2     478,840                            

In [81]:
# integrate temperature on monthly state-level basis
import time

noaa_url = "https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/{state}/tavg/1/0/2000-2025.json"
all_data = []

for state_id in range(1, 51):
    url = noaa_url.format(state=state_id)
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        j = r.json()
        for date, value in j["data"].items():
            year = int(date[:4])
            month = int(date[4:])
            all_data.append({
                "state_id": state_id,
                "year": year,
                "month": month,
                "temperature": value
            })

    except Exception as e:
        print(f"State {state_id} failed: {e}")

    time.sleep(0.4)

tmp = pd.DataFrame(all_data)
print(tmp.head())
print(tmp.shape)

State 49 failed: 404 Client Error: Not Found for url: https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/49/tavg/1/0/2000-2025.json
   state_id  year  month      temperature
0         1  2000      1  {'value': 46.5}
1         1  2000      2  {'value': 52.3}
2         1  2000      3  {'value': 58.9}
3         1  2000      4  {'value': 60.1}
4         1  2000      5    {'value': 74}
(15190, 4)


In [82]:
def extract_temp(val):
  return val['value']

tmp['temperature'] = tmp['temperature'].apply(extract_temp)

state_id_to_code = {
    1: "AL", 2: "AZ", 3: "AR", 4: "CA", 5: "CO",
    6: "CT", 7: "DE", 8: "FL", 9: "GA", 10: "HI",
    11: "ID", 12: "IL", 13: "IN", 14: "IA", 15: "KS",
    16: "KY", 17: "LA", 18: "ME", 19: "MD", 20: "MA",
    21: "MI", 22: "MN", 23: "MS", 24: "MO", 25: "MT",
    26: "NE", 27: "NV", 28: "NH", 29: "NJ", 30: "NM",
    31: "NY", 32: "NC", 33: "ND", 34: "OH", 35: "OK",
    36: "OR", 37: "PA", 38: "RI", 39: "SC", 40: "SD",
    41: "TN", 42: "TX", 43: "UT", 44: "VT", 45: "VA",
    46: "WA", 47: "WV", 48: "WI", 49: "WY", 50: "AK"
}

tmp['State'] = tmp['state_id'].map(state_id_to_code)
tmp.drop(columns=['state_id'], inplace=True)
tmp.head()

,year,month,temperature,State
0,2000,1,46.5,AL
1,2000,2,52.3,AL
2,2000,3,58.9,AL
3,2000,4,60.1,AL
4,2000,5,74.0,AL


In [83]:
tmp.rename(columns={'temperature':'temperature (F)','year':'Year','month':'Month','State':'state'}, inplace=True)
tmp['Year'] = tmp['Year'].astype('int32')
tmp['Month'] = tmp['Month'].astype('int32')
df = df.merge(tmp, on=['state', 'Year', 'Month'], how='inner')
print(df.head())
print(df.shape)

   State_Code     Period  Price  Year  Month state  \
0  F001242__3 2001-01-01  28.02  2001      1    PA   
1  F001242__3 2001-02-01  28.52  2001      2    PA   
2  F001242__3 2001-03-01  26.20  2001      3    PA   
3  F001242__3 2001-04-01  26.31  2001      4    PA   
4  F001242__3 2001-05-01  27.43  2001      5    PA   

                TYPE OF PRODUCER ENERGY SOURCE              (UNITS)  \
0  Total Electric Power Industry                Petroleum (Barrels)   
1  Total Electric Power Industry                Petroleum (Barrels)   
2  Total Electric Power Industry                Petroleum (Barrels)   
3  Total Electric Power Industry                Petroleum (Barrels)   
4  Total Electric Power Industry                Petroleum (Barrels)   

  CONSUMPTION  Treasury Yield Spread (10Y-2Y)  Inflation Rate (%)  \
0   1,005,652                            0.40            2.826171   
1     196,117                            0.44            2.826171   
2     478,840                            

In [92]:
prices = pd.read_csv('/content/Prices-Table 1.csv', skiprows=2)
prices.head()

,State,1970,1971,1972,1973,1974,1975,1976,1977,1978,...,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99,Unnamed: 100
0,AK,1.39,1.46,1.55,1.68,2.46,2.69,2.93,3.07,3.08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AL,1.37,1.51,1.61,1.84,2.50,2.83,3.21,3.76,3.99,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AR,1.51,1.65,1.69,1.81,2.49,2.96,3.29,3.81,4.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AZ,1.97,2.06,2.14,2.34,3.12,3.87,4.12,4.59,5.08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CA,1.74,1.81,1.91,2.13,3.03,3.47,3.80,4.27,4.58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
remove_colums = []
for col in prices.columns:
  if 'Unnamed' in col:
    remove_colums.append(col)

prices.drop(columns=remove_colums, inplace=True)

prices = prices.melt(
    id_vars=["State"],        # Columns to keep
    var_name="Year",          # Name for the new "variable" column
    value_name="Price"        # Name for the new "value" column
)

# Convert Year and Price to numeric
prices["Year"] = prices["Year"].astype(int)
prices["Price"] = pd.to_numeric(prices["Price"], errors='coerce')

print(prices.head())

  State  Year  Price
0    AK  1970   1.39
1    AL  1970   1.37
2    AR  1970   1.51
3    AZ  1970   1.97
4    CA  1970   1.74


In [95]:
prices.rename(columns={'Price':'Electricity Price ($ per million BTU)','State':'state'}, inplace=True)
prices['Year'] = prices['Year'].astype('int32')
df = df.merge(prices, on=['state', 'Year'], how='inner')
print(df.head())
print(df.shape)

   State_Code     Period  Price  Year  Month state  \
0  F001242__3 2001-01-01  28.02  2001      1    PA   
1  F001242__3 2001-02-01  28.52  2001      2    PA   
2  F001242__3 2001-03-01  26.20  2001      3    PA   
3  F001242__3 2001-04-01  26.31  2001      4    PA   
4  F001242__3 2001-05-01  27.43  2001      5    PA   

                TYPE OF PRODUCER ENERGY SOURCE              (UNITS)  \
0  Total Electric Power Industry                Petroleum (Barrels)   
1  Total Electric Power Industry                Petroleum (Barrels)   
2  Total Electric Power Industry                Petroleum (Barrels)   
3  Total Electric Power Industry                Petroleum (Barrels)   
4  Total Electric Power Industry                Petroleum (Barrels)   

  CONSUMPTION  Treasury Yield Spread (10Y-2Y)  Inflation Rate (%)  \
0   1,005,652                            0.40            2.826171   
1     196,117                            0.44            2.826171   
2     478,840                            

In [96]:
df.to_csv('/content/drive/MyDrive/agg_oil_data.csv', index=False)